In [1]:
import numpy as np
import rasterio
from pathlib import Path
import shutil

In [3]:
project_root = Path().resolve().parent.parent
train_dir =  project_root / "data" / "csen12" / "preprocess_test" / "training_dataset"/ "train_tiles_512_512"/"train_images"
# train_dir =  project_root / "data" / "venus" / "preprocessed"/ "tiles_512_512"/"train_images"
print(train_dir)

C:\skola\diplomka\data\csen12\preprocess_test\training_dataset\train_tiles_512_512\train_images


get number of bands

In [4]:
first_file = next(train_dir.glob("*.tif"))
with rasterio.open(first_file) as src:
    n_bands = src.count
    
print(n_bands)

7


In [5]:
sum_ = np.zeros(n_bands, dtype=np.float64)
sum_sq = np.zeros(n_bands, dtype=np.float64)
n_pixels = np.zeros(n_bands, dtype=np.int64)

In [7]:
# for each file in folder
for tif in train_dir.glob("*.tif"):
    # open as source
    with rasterio.open(tif) as src:
        # for each band
        for band in range(1, n_bands + 1):
            data = src.read(band)

            # create mask: ignore -1 and raster nodata if set
            mask = (data != -1)
            if src.nodata is not None:
                mask &= (data != src.nodata)

            # only valid data
            valid = data[mask].astype(np.float64)

            # sum of values per band
            sum_[band-1] += valid.sum()
            # sum squared per band
            sum_sq[band-1] += (valid ** 2).sum()
            # number of pixels per band
            n_pixels[band-1] += valid.size

# mean per band of all training tiles
mean = sum_ / n_pixels
# std per band of all training tiles
std = np.sqrt(sum_sq / n_pixels - mean**2)

# save for later use
np.save(train_dir.parent / "mean_orig.npy", mean)
np.save(train_dir.parent / "std_orig.npy", std)
# mean: [2648.48167567 2799.79738442 3401.88131586 3504.42423822 3773.07853654
#  4032.56518988 4174.64689501]
# std: [1474.61101007 1421.98926126 1637.653014   1643.86569117 1647.31244261
#  1707.59054954 1765.83208493]

print("mean:", mean)
print("std:", std)

mean: [2385.24184091 2700.87671661 3566.43299212 3762.44759587 4129.63952364
 4448.83055278 4656.44022751]
std: [711.02497195 697.93568421 880.93334747 825.81753817 802.86851403
 835.60696763 868.50570476]


In [4]:
data_dir = project_root / "data" / "csen12" / "training_dataset_3" / "good_quality_not_clear"
dst_dir = project_root / "data" / "csen12" / "training_dataset_3" / "good_quality_not_clear" / "standardized"
test_dir = project_root / "data" / "csen12" / "training_dataset_3" / "good_quality_not_clear" / "test_data_512_512"
mean = np.load(data_dir / "mean_s2_512_aug.npy")
std = np.load(data_dir / "std_s2_512_aug.npy")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\skola\\diplomka\\data\\csen12\\training_dataset_3\\good_quality_not_clear\\mean_s2_512_aug.npy'

In [11]:
def apply_fixed_zscore(images_dir: Path, masks_dir: Path, out_images_dir: Path, out_masks_dir: Path, mean: np.ndarray, std: np.ndarray):

    out_images_dir.mkdir(parents=True, exist_ok=True)
    out_masks_dir.mkdir(parents=True, exist_ok=True)

    for img_path in images_dir.glob("*.tif"):
        name = img_path.name

        mask_path = masks_dir / name

        with rasterio.open(img_path) as src:
            profile = src.profile
            data = src.read().astype(np.float32)

        for b in range(data.shape[0]):
            data[b] = (data[b] - mean[b]) / std[b]

        profile.update(dtype=rasterio.float32)

        with rasterio.open(out_images_dir / img_path.name, "w", **profile) as dst:
            dst.write(data)
            
        shutil.copy2(mask_path, out_masks_dir / mask_path.name)


In [12]:
mean = np.load(train_dir.parent / "mean_orig.npy")
std = np.load(train_dir.parent / "std_orig.npy")
apply_fixed_zscore(train_dir, train_dir.parent / "train_masks",  train_dir.parent /"standardized"/ "train_images",train_dir.parent /"standardized"/ "train_masks", mean, std)
apply_fixed_zscore(train_dir.parent / "val_images", train_dir.parent / "val_masks",  train_dir.parent /"standardized"/ "val_images",train_dir.parent /"standardized"/ "val_masks", mean, std)

In [8]:
apply_fixed_zscore(data_dir / "train_images",data_dir / "train_masks",  dst_dir / "train_images",dst_dir / "train_masks", mean, std)
apply_fixed_zscore(data_dir / "val_images",data_dir / "val_masks",  dst_dir / "val_images",dst_dir / "val_masks", mean, std)

In [9]:
apply_fixed_zscore(test_dir / "val_images",test_dir / "val_masks",  test_dir / "val_images_s",test_dir / "val_masks_s", mean, std)

In [10]:
print(dst_dir / "train_images")

C:\skola\diplomka\data\csen12\training_dataset_2\good_quality_not_clear\standardized\train_images


# venus

In [7]:
project_root = Path().resolve().parent.parent
test_dir = project_root / "data" / "venus"  / "preprocessed" / "test_data_512_512"
mean = np.load(test_dir / "standardized_by_s2t3_aug"/ "mean_s2_512_aug.npy")
std = np.load(test_dir / "standardized_by_s2t3_aug"/ "std_s2_512_aug.npy")
apply_fixed_zscore(test_dir / "val_images",test_dir / "val_masks",  test_dir / "standardized_by_s2t3_aug" / "val_images",test_dir / "standardized_by_s2t3_aug" / "val_masks", mean, std)

In [17]:
project_root = Path().resolve().parent.parent
test_dir = project_root / "data" / "venus"  / "preprocessed" / "test_data_512_512"
mean = np.load(test_dir.parent / "tiles_512_512" / "mean_venus.npy")
std = np.load(test_dir.parent / "tiles_512_512" / "std_venus.npy")
apply_fixed_zscore(test_dir / "val_images",test_dir / "val_masks",  test_dir / "standardized_by_venus" / "val_images",test_dir / "standardized_by_venus" / "val_masks", mean, std)